In [1]:
import re
import nltk
import joblib
import numpy as np
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from gensim import downloader as api

In [2]:
lemmatizer = nltk.stem.WordNetLemmatizer()

In [ ]:
MODEL_PATH = "../data/sentiment_classifier.joblib"
LABELS = {0 : "negative", 1: "positive"} 

def text_processor(text: str) -> str:
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r'<.+?>', '', text)

    # Remove URLs
    text = re.sub(r"[(http(s)?):\/\/(www\.)?a-zA-Z0-9@:%._\+~#=]{2,256}\.[a-z]{2,6}\b([-a-zA-Z0-9@:%_\+.~#?&//=]*)", "", text)

    # Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Stopword removal
    text = ' '.join([word for word in text.split() if word not in ENGLISH_STOP_WORDS and len(word) > 2])

    # Lemmatization
    text = lemmatizer.lemmatize(text)

    return text

classifier = joblib.load(MODEL_PATH)["classifier"]
google_w2v = api.load("word2vec-google-news-300")

def get_model(file_path :str):
    try:
        package = joblib.load(file_path)
        classifier = package["classifier"]
        vectorizer = package["vectorizer"]
    except Exception as e:
        print(f"Cannot load model files , error {e}")
        return None, None

    return classifier, vectorizer

def get_embeddings(tokens, model):
    vectors = [model[word] for word in tokens if word in model]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

def prediction_pipeline(
        user_input: str, 
        classifier=classifier, 
        model=google_w2v, 
        labels: dict = LABELS
    ) -> str:
    processed_text = text_processor(user_input)
    embedding = np.array([get_embeddings(processed_text.split(), model)])
    predicted_sentiment = classifier.predict(embedding)
    return labels[predicted_sentiment[0]].capitalize()

In [5]:
user_input = "Phil the Alien is one of those quirky films where the humour is based around the oddness of everything rather than actual punchlines.<br /><br />At first it was very odd and pretty funny but as the movie progressed I didn\'t find the jokes or oddness funny anymore.<br /><br />Its a low budget film (thats never a problem in itself), there were some pretty interesting characters, but eventually I just lost interest.<br /><br />I imagine this film would appeal to a stoner who is currently partaking.<br /><br />For something similar but better try 'Brother from another planet'"

In [ ]:
prediction_pipeline(user_input)

'Negative'